## Project 1: Language Modeling

In this project, you will implement several different types of language models for text.  We'll start with n-gram models, then move on to neural n-gram and LSTM language models.

**Warning: Do not start this project the day before it is due!**
Some parts require 20 minutes or more to run, so debugging and tuning can take a significant amount of time.

Our dataset for this project will be the WikiText2 language modeling dataset.  We provide some of the basic preprocessing, such as tokenization and rare word filtering (using the `<unk>` token).
Therefore, we can assume that all word types in the val/test set appear at least once in the training set.

In [1]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.3/474.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 17.0.0 which is incompatible.


In [2]:
# This block handles some imports and defines some constants.
# You shouldn't need to edit this, but if you want to
# import other standard python packages, that is fine.

# imports
from collections import Counter, defaultdict
import copy
import numpy as np
import math
import tqdm
import random
import pdb
from typing import List, Optional, Tuple, Union

from datasets import load_dataset
import torch
from torch import nn
import torch.nn.functional as F

# Some constants
UNK_TOK = "<unk>"
PAD_TOK = "<pad>"
EOS_TOK = "<eos>"

In [3]:
# This block defines the Vocabulary class we need later.
# You shouldn't need to edit this.

class Vocab:
    def __init__(self, train_text: List[str], min_freq=0):
        """
        We collect counts from train_text.
        train_text: a list of tokens.
        min_freq: if a token appears strictly less than this, it will not be
            added to vocab.
        """
        special_tokens = [UNK_TOK, PAD_TOK, EOS_TOK]

        counter = Counter(train_text)
        # Note that the order is fixed as long as the training text is the same.
        # it's sorted by frequency.
        all_tokens = [
            t for t, c in counter.most_common()
            if c >= min_freq and t not in special_tokens
        ]

        self.all_tokens = special_tokens + all_tokens
        self.str_to_id = {s: i for i, s in enumerate(self.all_tokens)}

        self.unk_tok = UNK_TOK
        self.pad_tok = PAD_TOK
        self.eos_tok = EOS_TOK

    def size(self) -> int:
        return len(self.all_tokens)


    def ids_to_strs(self, indices: List[int]) -> List[str]:
        return [self.all_tokens[ii] for ii in indices]


    def strs_to_ids(self, strings: List[str]) -> List[int]:
        return [self.str_to_id[s] for s in strings]


    def __contains__(self, token: str) -> bool:
        return token in self.str_to_id

In [4]:
# This block downloads and processes the data.
# You shouldn't need to edit this.

wikitext2_dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
print(f"Raw train examples: {wikitext2_dataset['train']['text'][:10]}")

# just use the simplest one for now
tokenizer = lambda x: x.split()

# tokenize datatsets
def preprocess(_dataset: List[str]) -> List[str]:
    """
    Each sentence in _dataset is tokenized into a list of strings.
    _dataset: List[str]. Each string is a sentence.
    """
    ret = []
    for sent in _dataset:
        sent = sent.rstrip('\n')
        # skip empty sentences
        if not sent:
            continue
        # add EOS to the end of sentence
        ret += tokenizer(sent) + [EOS_TOK]
    return ret

tok_train_dataset = preprocess(wikitext2_dataset['train']['text'])
tok_validation_dataset = preprocess(wikitext2_dataset['validation']['text'])
tok_test_dataset = preprocess(wikitext2_dataset['test']['text'])
print(f"Dataset size (#tokens) - Train: {len(tok_train_dataset)}; Validation: {len(tok_validation_dataset)}; Test: {len(tok_test_dataset)}.")

# build vocabulary: use `min_freq` to model UNK in training
### You'll need this vocab throughout this HW.
vocab = Vocab(tok_train_dataset, min_freq=2)
print(f"Vocab size: {vocab.size()}. Examples: {vocab.ids_to_strs(list(range(20)))}")

# handle UNKs properly
def replace_unseen_with_unk(_dataset: List[str]) -> List[str]:
    """
    We replace the unseen tokens in _dataset with vocab.unk_tok.
    """
    new_data = []
    for tok in _dataset:
        if tok in vocab:
            new_data.append(tok)
        else:
            new_data.append(vocab.unk_tok)
    return new_data

### You'll need these three datasets throughout this HW.
tok_train_dataset = replace_unseen_with_unk(tok_train_dataset)
tok_validation_dataset = replace_unseen_with_unk(tok_validation_dataset)
tok_test_dataset = replace_unseen_with_unk(tok_test_dataset)
print(f"Final train examples: {tok_train_dataset[:40]}")
print(f"Final val examples: {tok_validation_dataset[:40]}")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/733k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/6.36M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Raw train examples: ['', ' = Valkyria Chronicles III = \n', '', ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n', " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making

We've implemented a unigram model here as a demonstration.

In [5]:
class UnigramModel:
    def __init__(self, train_text: List[str]):
        self.counts = Counter(train_text)
        self.total_count = len(train_text)

    def probability(self, word: str) -> float:
        return self.counts[word] / self.total_count

    def next_word_probabilities(self, text_prefix: List[str]) -> List[str]:
        """
        Return a list of probabilities for each word in the vocabulary.
        In unigram model, `text_prefix` doesn't matter as we are not using any
            context at all.
        """
        return [self.probability(word) for word in vocab.all_tokens]

    def perplexity(self, full_text: List[str]) -> float:
        """Return the perplexity of the model on a text as a float.

        full_text -- a list of string tokens
        """
        log_probabilities = []
        for word in full_text:
            # Note that the base of the log doesn't matter
            # as long as the log and exp use the same base.
            log_probabilities.append(math.log(self.probability(word), 2))
        return 2 ** -np.mean(log_probabilities)

unigram_demonstration_model = UnigramModel(tok_train_dataset)
print('unigram validation perplexity:',
      unigram_demonstration_model.perplexity(tok_test_dataset))

unigram validation perplexity: 1057.2131456213988


In [6]:
def check_validity(model):
    """
    Performs several sanity checks on your model:
      1) That `next_word_probabilities` returns a valid distribution
      2) That perplexity matches a perplexity calculated from `next_word_probabilities`

    Although it is possible to calculate perplexity from `next_word_probabilities`,
      it is still good to have a separate more efficient method that only computes
      the probabilities of observed words.
    """

    log_probabilities = []
    for i in range(10):
        prefix = tok_validation_dataset[:i]
        probs = model.next_word_probabilities(prefix)
        assert min(probs) >= 0, "Negative value in next_word_probabilities"
        assert max(probs) <= 1 + 1e-8, "Value larger than 1 in next_word_probabilities"
        assert abs(sum(probs)-1) < 1e-4, "next_word_probabilities do not sum to 1"
        word_id = vocab.str_to_id[tok_validation_dataset[i]]
        selected_prob = probs[word_id]
        log_probabilities.append(math.log(selected_prob))


    perplexity = math.exp(-np.mean(log_probabilities))

    your_perplexity = model.perplexity(tok_validation_dataset[:10])
    assert abs(perplexity-your_perplexity) < 0.1, "your perplexity does not " + \
    "match the one we calculated from `next_word_probabilities`,\n" + \
    "at least one of `perplexity` or `next_word_probabilities` is incorrect.\n" + \
    f"we calcuated {perplexity} from `next_word_probabilities`,\n" + \
    f"but your perplexity function returned {your_perplexity} (on a small sample)."

In [7]:
check_validity(unigram_demonstration_model)

To generate from a language model, we can sample one word at a time conditioning on the words we have generated so far.

In [8]:
def generate_text(model, n=20, prefix=('<eos>', '<eos>')):
    prefix = list(prefix)
    for _ in range(n):
        probs = model.next_word_probabilities(prefix)
        word = random.choices(vocab.all_tokens, probs)[0]
        prefix.append(word)
    return ' '.join(prefix)

# unigram model does not utilize prefix
print(generate_text(unigram_demonstration_model, prefix=""))

arrows @-@ and styles but best method Hanuman was ground a of to mushrooms Neminatha also Aquitania their SO Azuma


TODO: Copy the printed output to your report.

In fact there are many strategies to get better-sounding samples, such as only sampling from the top-k words or sharpening the distribution with a temperature.  You can read more about sampling from a language model in this recent paper: https://arxiv.org/pdf/1904.09751.pdf.

You will need to submit some outputs from the models you implement for us to grade.  The following function will be used to generate the required output files.

In [9]:
!wget https://cal-cs288.github.io/sp21/project_files/proj_1/eval_prefixes.txt
!wget https://cal-cs288.github.io/sp21/project_files/proj_1/eval_output_vocab.txt
!wget https://cal-cs288.github.io/sp21/project_files/proj_1/eval_prefixes_short.txt
!wget https://cal-cs288.github.io/sp21/project_files/proj_1/eval_output_vocab_short.txt

def save_truncated_distribution(model, filename, short=True):
    """Generate a file of truncated distributions.

    Probability distributions over the full vocabulary are large,
    so we will truncate the distribution to a smaller vocabulary.

    Please do not edit this function
    """
    vocab_name = 'eval_output_vocab'
    prefixes_name = 'eval_prefixes'

    if short:
      vocab_name += '_short'
      prefixes_name += '_short'

    with open(f'{vocab_name}.txt', 'r') as eval_vocab_file:
        eval_vocab = [w.strip() for w in eval_vocab_file]
    eval_vocab_ids = sorted(list(set([vocab.str_to_id[s] if s in vocab else vocab.str_to_id[vocab.unk_tok]
                      for s in eval_vocab])))

    all_selected_probabilities = []
    with open(f'{prefixes_name}.txt', 'r') as eval_prefixes_file:
        lines = eval_prefixes_file.readlines()
        for line in tqdm.notebook.tqdm(lines, leave=False):
            prefix = line.strip().split(' ')
            probs = model.next_word_probabilities(prefix)
            selected_probs = np.array([probs[i] for i in eval_vocab_ids], dtype=np.float32)
            all_selected_probabilities.append(selected_probs)

    all_selected_probabilities = np.stack(all_selected_probabilities)
    np.save(filename, all_selected_probabilities)
    print('saved', filename)

--2024-09-25 21:09:36--  https://cal-cs288.github.io/sp21/project_files/proj_1/eval_prefixes.txt
Resolving cal-cs288.github.io (cal-cs288.github.io)... 185.199.109.153, 185.199.108.153, 185.199.111.153, ...
Connecting to cal-cs288.github.io (cal-cs288.github.io)|185.199.109.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 519055 (507K) [text/plain]
Saving to: ‘eval_prefixes.txt’

eval_prefixes.txt   100%[===================>] 506.89K  --.-KB/s    in 0.05s   

2024-09-25 21:09:36 (10.0 MB/s) - ‘eval_prefixes.txt’ saved [519055/519055]

--2024-09-25 21:09:36--  https://cal-cs288.github.io/sp21/project_files/proj_1/eval_output_vocab.txt
Resolving cal-cs288.github.io (cal-cs288.github.io)... 185.199.109.153, 185.199.108.153, 185.199.111.153, ...
Connecting to cal-cs288.github.io (cal-cs288.github.io)|185.199.109.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12497 (12K) [text/plain]
Saving to: ‘eval_output_vocab.txt’

eval_output_

In [10]:
save_truncated_distribution(unigram_demonstration_model,
                            'unigram_demonstration_predictions.npy')

  0%|          | 0/1000 [00:00<?, ?it/s]

saved unigram_demonstration_predictions.npy


### N-gram Model

Now it's time to implement an n-gram language model.

Because not every n-gram will have been observed in training, use add-alpha smoothing to make sure no output word has probability 0.

This is an example of bigram model with smoothing:
$$P(w_2|w_1)=\frac{C(w_1,w_2)+\alpha}{C(w_1)+N\alpha}$$

where $N$ is the vocab size and $C$ is the count for the given unigram/bigram.  An alpha value around `3e-3`  should work.  Later, we'll replace this smoothing with model backoff.

One **edge case** you will need to handle is at the beginning of the text where you don't have `n-1` prior words.  You may handle this by using a uniform distribution over the vocabulary.

A properly implemented bi-gram model should get a perplexity about/below **635** on the validation set.

**Note**: Do not change the signature of the `next_word_probabilities` and `perplexity` functions.  We will use these as a common interface for all of the different model types.  Make sure these two functions call `n_gram_probability`, because later we are going to override `n_gram_probability` in a subclass.
Also, we suggest pre-computing and caching the counts $C$ when you initialize `NGramModel` for efficiency.

In [17]:
class NGramModel:
    def __init__(self, train_text: List[str], n: int = 2, alpha: float = 3e-3):
        """
        Initializes the n-gram model by counting n-grams and (n-1)-grams from training data.

        train_text: List of words from the training data.
        n: The 'n' in 'n-gram', default is 2 (bigram).
        alpha: Smoothing parameter.
        """
        self.n = n
        self.alpha = alpha

        # Count occurrences of n-grams and (n-1)-grams
        self.ngram_counts = defaultdict(int)
        self.n_minus_1_counts = defaultdict(int)
        self.vocab = set(train_text)
        self.vocab_size = len(self.vocab)

        for i in range(len(train_text) - n + 1):
            ngram = tuple(train_text[i:i + n])
            n_minus_1_gram = tuple(train_text[i:i + n - 1])
            self.ngram_counts[ngram] += 1
            self.n_minus_1_counts[n_minus_1_gram] += 1

        # Cache for optimizing probability calculations
        self.prob_cache = {}
        self.context_prob_cache = {}


    def n_gram_probability(self, n_gram: Tuple[str, ...]) -> float:
        """
        Calculate the conditional probability of the last word in an n-gram.

        n_gram: A tuple of words representing an n-gram.
        Returns: The probability of the last word given the previous words in the n-gram.
        """
        if len(n_gram) != self.n:
            raise ValueError(f"Expected an n-gram of length {self.n}, but got {len(n_gram)}")

        if n_gram in self.prob_cache:
            return self.prob_cache[n_gram]

        if self.n == 1:
            # For unigram model
            unigram_count = self.ngram_counts[n_gram]
            total_unigrams = sum(self.ngram_counts.values())
            prob = (unigram_count + self.alpha) / (total_unigrams + self.alpha * self.vocab_size)
        else:
            # For n-gram model
            ngram_count = self.ngram_counts[n_gram]
            n_minus_1_gram = n_gram[:-1]
            n_minus_1_count = self.n_minus_1_counts.get(n_minus_1_gram, 0)
            prob = (ngram_count + self.alpha) / (n_minus_1_count + self.alpha * self.vocab_size)

        self.prob_cache[n_gram] = prob
        return prob

    def next_word_probabilities(self, text_prefix: List[str]) -> List[float]:
        """
        Given a text prefix, return a list of probabilities for each word in the vocabulary.

        text_prefix: A list of words as context.
        Returns: A list of probabilities corresponding to the likelihood of each word in the vocabulary.
        """
        if len(text_prefix) < self.n - 1:
            # If the prefix is too short, return uniform probabilities across the vocabulary
            return [1.0 / self.vocab_size] * self.vocab_size

        context = tuple(text_prefix[-(self.n - 1):])

        if context in self.context_prob_cache:
            return self.context_prob_cache[context]

        # Calculate probabilities for each word in the vocabulary
        probs = []
        for word in self.vocab:
            n_gram = context + (word,)
            prob = self.n_gram_probability(n_gram)
            probs.append(prob)

        self.context_prob_cache[context] = probs
        return probs

    def perplexity(self, full_text: List[str]) -> float:
        """
        Calculate the perplexity of the model on the provided text.

        full_text: A list of words representing the text.
        Returns: The perplexity score.
        """
        log_probabilities = []

        for i in range(len(full_text)):
            if i < self.n - 1:
                log_probabilities.append(math.log(1.0 / self.vocab_size))
            else:
                ngram = tuple(full_text[i - self.n + 1:i + 1])
                prob = self.n_gram_probability(ngram)
                log_probabilities.append(math.log(prob))

        avg_log_prob = -np.mean(log_probabilities)
        return math.exp(avg_log_prob)


In [12]:
unigram_model = NGramModel(tok_train_dataset, 1)
check_validity(unigram_model)
print('unigram validation perplexity:', unigram_model.perplexity(tok_validation_dataset)) # this should be the almost the same as our unigram model perplexity above

bigram_model = NGramModel(tok_train_dataset, n=2)
check_validity(bigram_model)
print('bigram validation perplexity:', bigram_model.perplexity(tok_validation_dataset))

trigram_model = NGramModel(tok_train_dataset, n=3)
check_validity(trigram_model)
print('trigram validation perplexity:', trigram_model.perplexity(tok_validation_dataset)) # this won't do very well...

unigram validation perplexity: 1096.2617610562029
bigram validation perplexity: 635.6280341733934
trigram validation perplexity: 4287.937643038851


In [22]:
save_truncated_distribution(bigram_model, 'bigram_predictions.npy') # this might take a few minutes

  0%|          | 0/1000 [00:00<?, ?it/s]

saved bigram_predictions.npy


Please download `bigram_predictions.npy` once you finish this section so that you can submit it.

In the block below, please report your bigram validation perplexity.  (We will use this to help us calibrate our scoring on the test set.)

TODO: Report the perplexity in your report.

Bigram validation perplexity: ***fill in here***

We can also generate samples from the model to get an idea of how it is doing.

In [23]:
print(generate_text(bigram_model))

<eos> <eos> DuVall replied 1214 Daelen therapy session Missile factors tamp lobes sandbanks MB motorsport Trujillo Hundred Mottram Convocation welfare brisk 369


We now free up some RAM, **it is important to run the cell below, otherwise you will likely run out of RAM in the Colab runtime.**

In [24]:
# Free up some RAM.
del bigram_model
del trigram_model

This basic model works okay for bigrams, but a better strategy (especially for higher-order models) is to use backoff.  Implement backoff with absolute discounting.
$$P\left(w_i|w_{i-n+1}^{i-1}\right)=\frac{max\left\{C(w_{i-n+1}^i)-\delta,0\right\}}{\sum_{w_i} C(w_{i-n+1}^i)} + \alpha(w_{i-n+1}^{i-1}) P(w_i|w_{i-n+2}^{i-1})$$

$$\alpha\left(w_{i-n+1}^{i-1}\right)=\frac{\delta N_{1+}(w_{i-n+1}^{i-1})}{{\sum_{w_i} C(w_{i-n+1}^i)}}$$
where $N_{1+}$ is the number of words that appear after the previous $n-1$ words (the number of times the max will select something other than 0 in the first equation).  If $\sum_{w_i} C(w_{i-n+1}^i)=0$, use the lower order model probability directly (the above equations would have a division by 0).

We found a discount $\delta$ of 0.9 to work well based on validation performance.  A trigram model with this discount value should get a validation perplexity around/below **310**.

In [19]:
class DiscountBackoffModel(NGramModel):
    def __init__(self, train_text: List[str],
                 lower_order_model: Union[NGramModel, "DiscountBackoffModel"],
                 n: int = 2,
                 delta: float = 0.9):
        """
        Initialize the Discount Backoff Model. This model uses backoff to a lower-order n-gram model
        when higher-order contexts have sparse data.

        train_text: The training data as a list of tokens.
        lower_order_model: A lower-order n-gram model for backoff purposes.
        n: The 'n' in 'n-gram', must be >= 2.
        delta: The discount applied to the n-gram counts for smoothing.
        """
        assert n >= 2, "n must be at least 2 for backoff models."
        super().__init__(train_text, n=n)
        self.lower_order_model = lower_order_model
        self.discount = delta


        self.ngram_counts = defaultdict(int)
        self.context_counts = defaultdict(int)
        self.unique_followers = defaultdict(set)


        padded_text = [vocab.eos_tok] * (self.n - 1) + train_text


        for i in range(len(padded_text) - self.n + 1):
            ngram = tuple(padded_text[i:i + self.n])
            context = ngram[:-1]
            word = ngram[-1]

            self.ngram_counts[ngram] += 1
            self.context_counts[context] += 1
            self.unique_followers[context].add(word)


        self.prob_cache = {}

    def n_gram_probability(self, n_gram: Tuple[str, ...]) -> float:
        """
        Calculate the probability of the last word in an n-gram using discounting and backoff.

        n_gram: A tuple representing an n-gram.
        Returns: The conditional probability of the last word given the preceding context.
        """
        assert len(n_gram) == self.n, f"Expected an n-gram of length {self.n}, but got {len(n_gram)}"


        if n_gram in self.prob_cache:
            return self.prob_cache[n_gram]

        context = n_gram[:-1]
        word = n_gram[-1]


        ngram_count = self.ngram_counts.get(n_gram, 0)
        context_count = self.context_counts.get(context, 0)

        if context_count == 0:
            prob = self.lower_order_model.n_gram_probability(n_gram[1:])
            self.prob_cache[n_gram] = prob
            return prob


        discounted_count = max(ngram_count - self.discount, 0)
        ngram_prob = discounted_count / context_count


        unique_followers_count = len(self.unique_followers[context])
        alpha = (self.discount * unique_followers_count) / context_count


        lower_order_ngram = n_gram[1:]
        backoff_prob = self.lower_order_model.n_gram_probability(lower_order_ngram)


        prob = ngram_prob + alpha * backoff_prob


        self.prob_cache[n_gram] = prob
        return prob


In [14]:

bigram_backoff_model = DiscountBackoffModel(tok_train_dataset, unigram_model, 2)
check_validity(bigram_backoff_model)
print('bigram backoff validation perplexity:', bigram_backoff_model.perplexity(tok_validation_dataset))

trigram_backoff_model = DiscountBackoffModel(tok_train_dataset, bigram_backoff_model, 3)
check_validity(trigram_backoff_model)
print('trigram backoff validation perplexity:', trigram_backoff_model.perplexity(tok_validation_dataset))

bigram backoff validation perplexity: 346.447223388478
trigram backoff validation perplexity: 310.72627114846125


In [15]:
save_truncated_distribution(trigram_backoff_model, 'trigram_pbackoff_redictions.npy') # this might take a few minutes

  0%|          | 0/1000 [00:00<?, ?it/s]

saved trigram_pbackoff_redictions.npy


TODO: Report your trigram backoff model perplexity.

Trigram backoff validation perplexity: ***fill in here***

Free up RAM.

In [20]:
# Release models we don't need any more.
del unigram_model
del bigram_backoff_model
del trigram_backoff_model

### Neural N-gram Model

In this section, you will implement a neural version of an n-gram model.  The model will use a simple feedforward neural network that takes the previous `n-1` words and outputs a distribution over the next word.

You will use PyTorch to implement the model.  We've provided a little bit of code to help with the data loading using PyTorch's data loaders (https://pytorch.org/docs/stable/data.html)

A model with the following architecture and hyperparameters should reach a validation perplexity around/below **240**.
* embed the words with dimension 128, then flatten into a single embedding for $n-1$ words (with size $(n-1)*128$)
* run 2 hidden layers with 1024 hidden units, then project down to size 128 before the final layer (ie. 4 layers total).
* use weight tying for the embedding and final linear layer (this made a very large difference in our experiments); you can do this by creating the output layer with `nn.Linear`, then using `F.embedding` with the linear layer's `.weight` to embed the input
* rectified linear activation (ReLU) and dropout 0.1 after first 2 hidden layers. **Note: You will likely find a performance drop if you add a nonlinear activation function after the dimension reduction layer.**
* train for 10 epochs with the Adam optimizer (should take around 15-20 minutes)
* do early stopping based on validation set perplexity.


We encourage you to try other architectures and hyperparameters, and you will likely find some that work better than the ones listed above.  A proper implementation with these should be enough to receive full credit on the assignment, though.

In [21]:
class NeuralNgramDataset(torch.utils.data.Dataset):
    def __init__(self, text_token_ids: List[int], n: int):
        self.text_token_ids = text_token_ids
        self.n = n

    def __len__(self):
        return len(self.text_token_ids)

    def __getitem__(self, i: int):
        if i < self.n - 1:
            prev_token_ids = [vocab.str_to_id[vocab.eos_tok]] * (self.n - i - 1) + \
                              self.text_token_ids[:i]
        else:
            prev_token_ids = self.text_token_ids[i - self.n + 1 : i]

        assert len(prev_token_ids) == self.n - 1, prev_token_ids

        x = torch.tensor(prev_token_ids, dtype=torch.long)
        y = torch.tensor(self.text_token_ids[i], dtype=torch.long)
        return x, y

class NeuralNGramNetwork(nn.Module):
    # a PyTorch Module that holds the neural network for your model

    def __init__(
            self, n: int,
            embed_dim: int = 128,
            hidden_dim: int = 1024,
            dropout_rate: float = 0.1
        ):
        super().__init__()
        self.n = n

        # YOUR CODE HERE
        vocab_size = len(vocab.str_to_id)
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear((n - 1) * embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

        # Activation and dropout
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)

        pass


    def forward(self, x):
        # x is a tensor of inputs with shape (batch, n-1)
        # this function returns a tensor of log probabilities with shape (batch, vocab_size)

        # YOUR CODE HERE
        embeddings = self.embedding(x)  # Shape: (batch_size, n-1, embed_dim)

        flattened = embeddings.view(embeddings.size(0), -1)  # Shape: (batch_size, (n-1) * embed_dim)

        hidden = self.relu(self.fc1(flattened))
        hidden = self.dropout(hidden)
        hidden = self.relu(self.fc2(hidden))
        hidden = self.dropout(hidden)

        logits = self.output_layer(hidden)  # Shape: (batch_size, vocab_size)

        return F.log_softmax(logits, dim=-1)
        pass



class NeuralNGramModel:
    # a class that wraps NeuralNGramNetwork to handle training and evaluation
    # it's ok if this doesn't work for unigram modeling
    def __init__(self, n: int, device: str = "cpu", **model_configs):
        self.n = n
        self.device = device

        if "cuda" in self.device:
            assert torch.cuda.is_available(), "no GPU found, in Colab go to 'Edit->Notebook settings' and choose a GPU hardware accelerator"

        self.network = NeuralNGramNetwork(n, **model_configs).to(self.device)

    def train(
        self,
        n_epoch: int = 10, lr: float = 0.001, batch_size: int = 128
    ):
        train_dataset = NeuralNgramDataset(vocab.strs_to_ids(tok_train_dataset), self.n)
        train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        # iterating over train_dataloader with a for loop will return a 2-tuple of batched tensors
        # the first tensor will be previous token ids with size (batch, n-1),
        # and the second will be the current token id with size (batch, )
        # you will need to move these tensors to GPU, e.g. by using the Tensor.to() function.

        # this will take some time to run; use tqdm.notebook.tqdm to get a progress bar

        # YOUR CODE HERE
        optimizer = torch.optim.Adam(self.network.parameters(), lr=lr)
        criterion = nn.NLLLoss()

        # Early stopping parameters
        best_val_perplexity = float('inf')
        patience = 2
        wait = 0

        # Training loop
        for epoch in range(n_epoch):
            self.network.train()
            total_loss = 0

            for x_batch, y_batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{n_epoch}"):
                x_batch = x_batch.to(self.device)
                y_batch = y_batch.to(self.device)

                optimizer.zero_grad()

                # Forward pass
                log_probs = self.network(x_batch)
                loss = criterion(log_probs, y_batch)

                # Backward pass and optimization
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            avg_loss = total_loss / len(train_dataloader)
            print(f'Epoch {epoch + 1}, Training Loss: {avg_loss:.4f}')

            # Validation perplexity
            val_perplexity = self.perplexity(tok_validation_dataset)
            print(f'Epoch {epoch + 1}, Validation Perplexity: {val_perplexity:.2f}')

            # Early stopping logic
            if val_perplexity < best_val_perplexity:
                best_val_perplexity = val_perplexity
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    print("Early stopping due to no improvement.")
                    break

        pass

    def next_word_probabilities(self, text_prefix: List[str]) -> List[float]:
        # YOUR CODE HERE
        # Don't forget self.network.eval().
        # You will need to convert text_prefix from strings to numbers with the `vocab.strs_to_ids` function.
        # If your `perplexity` function below is based on a NeuralNgramDataset DataLoader, you will need to use the same strategy for prefixes with less than n-1 tokens to pass the validity check.
        # The data loader appends extra "<eos>" (end of sentence) tokens to the start of the input so there are always enough to run the network
        self.network.eval()
        with torch.no_grad():
            ids_prefix = vocab.strs_to_ids(text_prefix)

            if len(ids_prefix) < self.n - 1:
                padding = [vocab.str_to_id[vocab.eos_tok]] * (self.n - 1 - len(ids_prefix))
                ids_prefix = padding + ids_prefix
            else:
                ids_prefix = ids_prefix[-(self.n - 1):]

            x_tensor = torch.tensor([ids_prefix], dtype=torch.long).to(self.device)

            log_probs = self.network(x_tensor)

            probs = torch.exp(log_probs[0]).cpu().numpy()

            return probs
        pass

    def perplexity(self, text: List[str]) -> float:
        # You may want to use a DataLoader here with a NeuralNgramDataset
        # Don't forget self.network.eval()

        # YOUR CODE HERE
        self.network.eval()
        with torch.no_grad():
            token_ids = vocab.strs_to_ids(text)
            dataset = NeuralNgramDataset(token_ids, self.n)
            loader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=False)

            total_log_prob = 0
            total_count = 0

            for x_batch, y_batch in loader:
                x_batch = x_batch.to(self.device)
                y_batch = y_batch.to(self.device)

                # Forward pass
                log_probs = self.network(x_batch)

                # Gather log probabilities for the target words
                log_probs_target = log_probs.gather(1, y_batch.unsqueeze(1)).squeeze(1)

                total_log_prob += log_probs_target.sum().item()
                total_count += y_batch.size(0)

            avg_log_prob = total_log_prob / total_count
            perplexity = math.exp(-avg_log_prob)

            return perplexity
        pass

In [ ]:
# it's probabily better to first debug with cpu so you don't waste the limited GPU time.
# then you use device="cuda" to train on GPU.
neural_trigram_model = NeuralNGramModel(3, device="cuda")
check_validity(neural_trigram_model)
neural_trigram_model.train(lr=5e-4)
print('neural trigram validation perplexity:', neural_trigram_model.perplexity(tok_validation_dataset))

Epoch 1/10:   0%|          | 0/16217 [00:00<?, ?it/s]

Epoch 1, Training Loss: 6.1006
Epoch 1, Validation Perplexity: 275.51


Epoch 2/10:   0%|          | 0/16217 [00:00<?, ?it/s]

Epoch 2, Training Loss: 5.5203
Epoch 2, Validation Perplexity: 243.50


Epoch 3/10:   0%|          | 0/16217 [00:00<?, ?it/s]

Epoch 3, Training Loss: 5.2791
Epoch 3, Validation Perplexity: 232.82


Epoch 4/10:   0%|          | 0/16217 [00:00<?, ?it/s]

Epoch 4, Training Loss: 5.1146
Epoch 4, Validation Perplexity: 234.76


Epoch 5/10:   0%|          | 0/16217 [00:00<?, ?it/s]

Epoch 5, Training Loss: 4.9899
Epoch 5, Validation Perplexity: 240.38
Early stopping
neural trigram validation perplexity: 240.37557788213746


In [ ]:
save_truncated_distribution(neural_trigram_model, 'neural_trigram_predictions.npy', short=False)

  0%|          | 0/5000 [00:00<?, ?it/s]

saved neural_trigram_predictions.npy


TODO: Fill in your neural trigram perplexity in the report.

<!-- Do not remove this comment, it is used by the autograder: RqYJKsoTS6 -->

Neural trigram validation perplexity: ***fill in here***

Free up RAM.

In [ ]:
# Delete model we don't need.
del neural_trigram_model

### LSTM Model

For this stage of the project, you will implement an LSTM language model.

For recurrent language modeling, the data batching strategy is a bit different from what is used in some other tasks.  Sentences are concatenated together so that one sentence starts right after the other, and an unfinished sentence will be continued in the next batch.
To properly deal with this input format, you should **save the last state of the LSTM from a batch to feed in as the first state of the next batch**.  When you save state across different batches, you should call `.detach()` on the state tensors before the next batch to tell PyTorch not to backpropagate gradients through the state into the batch you have already finished (which will cause a runtime error).

We expect your model to reach a validation perplexity around/below **214**.
The following architecture and hyperparameters should be sufficient to get there.
* 3 LSTM layers with 512 units
* dropout of 0.5 after each LSTM layer
* instead of projecting directly from the last LSTM output to the vocabulary size for softmax, project down to a smaller size first (e.g. 512->128->vocab_size). **NOTE: You may find that adding nonlinearities between these layers can hurt performance, try without first.**
* use the same weights for the embedding layer and the pre-softmax layer; dimension 128
* train with Adam (using default learning rates) for at least 20 epochs


In [24]:
# ref: https://github.com/pytorch/text/blob/0.5.0/torchtext/data/iterator.py#L173

class LstmDataIterator:
    def __init__(self, dataset: List[int], batch_size: int = 64, seq_len: int = 32, device: str = "cpu"):
        self.batch_size = batch_size  # number of sequence
        self.seq_len = seq_len  # length of each sequence
        self.device = device  # CPU or GPU

        # pad the dataset so that it is divisible by batch_size
        dataset = dataset + [vocab.str_to_id[vocab.pad_tok]] * (math.ceil(len(dataset) / batch_size) * batch_size - len(dataset)) # padding done to make it divisible by btach_size

        self.n_samples = math.ceil(
            (len(dataset) // batch_size - 1) / seq_len       # calculate no. of batches from padded dataset
        )

        dataset = torch.tensor(dataset, dtype=torch.long)
        self.dataset = dataset.view(batch_size, -1).t().contiguous()  # reshaping of dataset into 2-D tensor

    def __len__(self):
        return self.n_samples    # return no. of samples

    def __getitem__(self, i: int):
        start = i * self.seq_len
        end = min(start + self.seq_len, self.dataset.shape[0] - 1)

        inputs = self.dataset[start : end]
        outputs = self.dataset[start + 1 : end + 1]
        assert inputs.shape == outputs.shape, f"{i}: {inputs.shape} {outputs.shape}"
        # tensor shape (seq_len, batch_size)
        return inputs.to(self.device), outputs.to(self.device)

In [23]:
class LSTMNetwork(nn.Module):
    # Neural network for the LSTM language model

    def __init__(self, embed_dim=128, n_layer=3, hidden_dim=512, dropout_rate=0.5):
        super(LSTMNetwork, self).__init__()

        # YOUR CODE HERE

        vocab_size = len(vocab.str_to_id)
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layer, dropout=dropout_rate)

        self.fc = nn.Linear(hidden_dim, embed_dim)                      # Dimension reduction layer (hidden_dim -> embed_dim)

        self.dropout = nn.Dropout(dropout_rate)

        pass

    def forward(self, x: torch.Tensor, state: Optional[Tuple[torch.Tensor, torch.Tensor]] = None):
        """
        x - Tensor of input token IDs with shape (seq_len, batch_size)
        state - Tuple of (hidden_state, cell_state), each with shape (num_layers, batch_size, hidden_dim)
        Returns:
          - log_probs: Tensor of log probabilities with shape (seq_len, batch_size, vocab_size)
          - state: Updated LSTM state tuple
        """

        # YOUR CODE HERE

        embeddings = self.embedding(x)  # Shape: (seq_len, batch_size, embed_dim)

        lstm_out, state = self.lstm(embeddings, state)  # lstm_out shape: (seq_len, batch_size, hidden_dim)

        lstm_out = self.dropout(lstm_out)

        reduced = self.fc(lstm_out)  # Shape: (seq_len, batch_size, embed_dim)

        logits = torch.matmul(reduced, self.embedding.weight.t())  # Shape: (seq_len, batch_size, vocab_size)

        log_probs = F.log_softmax(logits, dim=-1)

        return log_probs, state

        pass


class LSTMModel:
    "Wrapper class for training and evaluating the LSTM model."

    def __init__(self, device: str = "cpu", **model_configs):
        self.device = device
        if "cuda" in self.device:
            assert torch.cuda.is_available(), "No GPU found. Please check your device settings."

        self.network = LSTMNetwork(**model_configs).to(self.device)

    def train(self, n_epoch: int = 20, lr: float = 1e-3, batch_size: int = 64, seq_len: int = 32):
        # Fetch the training data
        train_ids = vocab.strs_to_ids(tok_train_dataset)
        train_data_iter = LstmDataIterator(train_ids, batch_size, seq_len, self.device)

        # YOUR CODE HERE

        optimizer = torch.optim.Adam(self.network.parameters(), lr=lr)
        criterion = nn.NLLLoss(ignore_index=vocab.str_to_id[vocab.pad_tok])

        for epoch in range(n_epoch):
            total_loss = 0
            state = None  # Initialize hidden state

            self.network.train()

            for i in range(len(train_data_iter)):
                inputs, targets = train_data_iter[i]

                if state is not None:
                    state = tuple(s.detach() for s in state)

                optimizer.zero_grad()

                log_probs, state = self.network(inputs, state)         # forward pass

                log_probs = log_probs.view(-1, log_probs.size(-1))
                targets = targets.view(-1)

                loss = criterion(log_probs, targets)

                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            avg_loss = total_loss / len(train_data_iter)
            print(f'Epoch {epoch + 1}/{n_epoch}, Loss: {avg_loss:.4f}')

        pass

    def next_word_probabilities(self, text_prefix: List[str]):
        "Return a list of probabilities for the next word in the vocabulary."

        # YOUR CODE HERE

        self.network.eval()
        with torch.no_grad():
            ids_prefix = vocab.strs_to_ids(text_prefix)
            x = torch.tensor(ids_prefix, dtype=torch.long, device=self.device).unsqueeze(1)  # Shape: (seq_len, 1)

            log_probs, _ = self.network(x)          # forward pass

            last_log_probs = log_probs[-1, 0]  # Shape: (vocab_size,)
            probs = torch.exp(last_log_probs).cpu().numpy()

        return probs

        pass

    def dataset_perplexity(self, dataset: List[str], batch_size: int = 64, seq_len: int = 32):
        "Compute the perplexity of the model on the given dataset."

        data_ids = vocab.strs_to_ids(dataset)
        data_iter = LstmDataIterator(data_ids, batch_size, seq_len, self.device)

        # YOUR CODE HERE

        total_loss = 0
        total_tokens = 0
        criterion = nn.NLLLoss(ignore_index=vocab.str_to_id[vocab.pad_tok], reduction='sum')
        state = None

        self.network.eval()
        with torch.no_grad():
            for i in range(len(data_iter)):
                inputs, targets = data_iter[i]

                log_probs, state = self.network(inputs, state)          # forward pass

                state = tuple(s.detach() for s in state)

                log_probs = log_probs.view(-1, log_probs.size(-1))
                targets = targets.view(-1)

                loss = criterion(log_probs, targets)
                total_loss += loss.item()

                mask = targets != vocab.str_to_id[vocab.pad_tok]
                total_tokens += mask.sum().item()

        avg_loss = total_loss / total_tokens
        perplexity = math.exp(avg_loss)
        return perplexity

        pass

In [11]:
lstm_model = LSTMModel(device="cuda")
lstm_model.train()

print('lstm validation perplexity:', lstm_model.dataset_perplexity(tok_validation_dataset))

Epoch 1/20, Loss: 7.0453
Epoch 2/20, Loss: 6.3030
Epoch 3/20, Loss: 6.0378
Epoch 4/20, Loss: 5.8416
Epoch 5/20, Loss: 5.6829
Epoch 6/20, Loss: 5.5526
Epoch 7/20, Loss: 5.4471
Epoch 8/20, Loss: 5.3605
Epoch 9/20, Loss: 5.2854
Epoch 10/20, Loss: 5.2207
Epoch 11/20, Loss: 5.1625
Epoch 12/20, Loss: 5.1127
Epoch 13/20, Loss: 5.0665
Epoch 14/20, Loss: 5.0243
Epoch 15/20, Loss: 4.9838
Epoch 16/20, Loss: 4.9474
Epoch 17/20, Loss: 4.9139
Epoch 18/20, Loss: 4.8826
Epoch 19/20, Loss: 4.8533
Epoch 20/20, Loss: 4.8250
lstm validation perplexity: 156.40791950612498


In [12]:
save_truncated_distribution(lstm_model, 'lstm_predictions.npy', short=False)

  0%|          | 0/5000 [00:00<?, ?it/s]

saved lstm_predictions.npy


TODO: Report your LSTM perplexity.

LSTM validation perplexity: ***fill in here***

# Experimentation: 1-Page Report

Now it's time for you to experiment.  Try to reach a validation perplexity below 120. You may either modify the LSTM class above, or copy it down to the code cell below and modify it there. Just **be sure to run code cell below to generate results with your improved LSTM**.  

It is okay if the bulk of your improvements are due to hyperparameter tuning (such as changing number or sizes of layers), but implement at least one more substantial change to the model.  Here are some ideas (several of which come from https://arxiv.org/pdf/1708.02182.pdf):
* activation regularization - add a l2 regularization penalty on the activation of the LSTM output (standard l2 regularization is on the weights)
* weight-drop regularization - apply dropout to the weight matrices instead of activations
* learning rate scheduling - decrease the learning rate during training
* embedding dropout - zero out the entire embedding for a random set of words in the embedding matrix
* ensembling - average the predictions of several models trained with different initialization random seeds
* temporal activation regularization - add l2 regularization on the difference between the LSTM output activations at adjacent timesteps

You may notice that most of these suggestions are regularization techniques.  This dataset is considered fairly small, so regularization is one of the best ways to improve performance.

TODO: In the report, submit a write-up describing the extensions and/or modifications that you tried.  Your description should be **1-page maximum** in length.
For full credit, your write-up should include:
1.   A concise and precise description of the extension that you tried.
2.   A motivation for why you believed this approach might improve your model.
3.   A discussion of whether the extension was effective and/or an analysis of the results.  This will generally involve some combination of tables, learning curves, etc.
4.   A bottom-line summary of your results comparing validation perplexities of your improvement to the original LSTM.


Run the cell below in order to train your improved LSTM and evaluate it.  

In [22]:
## Feel free to copy your original LSTM solution down here to modify for your report if you'd like.
# YOUR CODE [optionally] HERE
class LSTMNetwork(nn.Module):
    # Neural network for the LSTM language model

    def __init__(self, embed_dim=256, n_layer=4, hidden_dim=1024, dropout_rate=0.5):
        super(LSTMNetwork, self).__init__()

        # YOUR CODE HERE

        vocab_size = len(vocab.str_to_id)
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.embedding_dropout = nn.Dropout(dropout_rate)

        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layer, dropout=dropout_rate)

        self.fc = nn.Linear(hidden_dim, embed_dim)

        self.dropout = nn.Dropout(dropout_rate)

        pass

    def forward(self, x: torch.Tensor, state: Optional[Tuple[torch.Tensor, torch.Tensor]] = None):
        """
        x - Tensor of input token IDs with shape (seq_len, batch_size)
        state - Tuple of (hidden_state, cell_state), each with shape (num_layers, batch_size, hidden_dim)
        Returns:
          - log_probs: Tensor of log probabilities with shape (seq_len, batch_size, vocab_size)
          - state: Updated LSTM state tuple
        """

        # YOUR CODE HERE

        embeddings = self.embedding(x)  # Shape: (seq_len, batch_size, embed_dim)
        embeddings = self.embedding_dropout(embeddings)

        lstm_out, state = self.lstm(embeddings, state)  # lstm_out shape: (seq_len, batch_size, hidden_dim)

        lstm_out = self.dropout(lstm_out)

        reduced = self.fc(lstm_out)  # Shape: (seq_len, batch_size, embed_dim)

        logits = torch.matmul(reduced, self.embedding.weight.t())  # Shape: (seq_len, batch_size, vocab_size)

        log_probs = F.log_softmax(logits, dim=-1)

        return log_probs, state

        pass


class LSTMModel:
    "Wrapper class for training and evaluating the LSTM model."

    def __init__(self, device: str = "cpu", **model_configs):
        self.device = device
        if "cuda" in self.device:
            assert torch.cuda.is_available(), "No GPU found. Please check your device settings."

        self.network = LSTMNetwork(**model_configs).to(self.device)

    def train(self, n_epoch: int = 20, lr: float = 1e-3, batch_size: int = 64, seq_len: int = 64):
        # Fetch the training data
        train_ids = vocab.strs_to_ids(tok_train_dataset)
        train_data_iter = LstmDataIterator(train_ids, batch_size, seq_len, self.device)

        # YOUR CODE HERE

        optimizer = torch.optim.Adam(self.network.parameters(), lr=lr, weight_decay=1e-5)
        criterion = nn.NLLLoss(ignore_index=vocab.str_to_id[vocab.pad_tok])

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=2)

        for epoch in range(n_epoch):
            total_loss = 0
            state = None  # Initialize hidden state

            self.network.train()

            for i in range(len(train_data_iter)):
                inputs, targets = train_data_iter[i]

                if state is not None:
                    state = tuple(s.detach() for s in state)

                optimizer.zero_grad()

                log_probs, state = self.network(inputs, state)

                log_probs = log_probs.view(-1, log_probs.size(-1))
                targets = targets.view(-1)

                loss = criterion(log_probs, targets)

                activation_loss = 1e-6 * torch.sum(torch.pow(state[0], 2)) / state[0].numel()
                loss += activation_loss

                loss.backward()

                torch.nn.utils.clip_grad_norm_(self.network.parameters(), max_norm=5)

                optimizer.step()

                total_loss += loss.item()

            avg_loss = total_loss / len(train_data_iter)
            print(f'Epoch {epoch + 1}/{n_epoch}')

            val_perplexity = self.dataset_perplexity(tok_validation_dataset)
            scheduler.step(val_perplexity)
            print(f'Validation Perplexity after Epoch {epoch + 1}: {val_perplexity:.2f}')

        pass

    def next_word_probabilities(self, text_prefix: List[str]):
        "Return a list of probabilities for the next word in the vocabulary."

        # YOUR CODE HERE

        self.network.eval()
        with torch.no_grad():
            ids_prefix = vocab.strs_to_ids(text_prefix)
            x = torch.tensor(ids_prefix, dtype=torch.long, device=self.device).unsqueeze(1)  # Shape: (seq_len, 1)

            log_probs, _ = self.network(x)

            last_log_probs = log_probs[-1, 0]  # Shape: (vocab_size,)
            probs = torch.exp(last_log_probs).cpu().numpy()

        return probs

        pass

    def dataset_perplexity(self, dataset: List[str], batch_size: int = 64, seq_len: int = 64):
        "Compute the perplexity of the model on the given dataset."

        data_ids = vocab.strs_to_ids(dataset)
        data_iter = LstmDataIterator(data_ids, batch_size, seq_len, self.device)

        # YOUR CODE HERE

        total_loss = 0
        total_tokens = 0
        criterion = nn.NLLLoss(ignore_index=vocab.str_to_id[vocab.pad_tok], reduction='sum')
        state = None

        self.network.eval()
        with torch.no_grad():
            for i in range(len(data_iter)):
                inputs, targets = data_iter[i]

                log_probs, state = self.network(inputs, state)

                if state is not None:
                    state = tuple(s.detach() for s in state)

                log_probs = log_probs.view(-1, log_probs.size(-1))
                targets = targets.view(-1)

                loss = criterion(log_probs, targets)
                total_loss += loss.item()

                mask = targets != vocab.str_to_id[vocab.pad_tok]
                total_tokens += mask.sum().item()

        avg_loss = total_loss / total_tokens
        perplexity = math.exp(avg_loss)
        return perplexity

        pass
##

In [19]:

lstm_model = LSTMModel(device="cuda")
lstm_model.train()

print('lstm validation perplexity:', lstm_model.dataset_perplexity(tok_validation_dataset))
save_truncated_distribution(lstm_model, 'lstm_predictions.npy', short=False)

Epoch 1/20
Validation Perplexity after Epoch 1: 571.20
Epoch 2/20
Validation Perplexity after Epoch 2: 398.22
Epoch 3/20
Validation Perplexity after Epoch 3: 308.71
Epoch 4/20
Validation Perplexity after Epoch 4: 245.92
Epoch 5/20
Validation Perplexity after Epoch 5: 210.58
Epoch 6/20
Validation Perplexity after Epoch 6: 186.34
Epoch 7/20
Validation Perplexity after Epoch 7: 169.57
Epoch 8/20
Validation Perplexity after Epoch 8: 158.46
Epoch 9/20
Validation Perplexity after Epoch 9: 149.14
Epoch 10/20
Validation Perplexity after Epoch 10: 141.80
Epoch 11/20
Validation Perplexity after Epoch 11: 137.06
Epoch 12/20
Validation Perplexity after Epoch 12: 132.37
Epoch 13/20
Validation Perplexity after Epoch 13: 128.43
Epoch 14/20
Validation Perplexity after Epoch 14: 125.49
Epoch 15/20
Validation Perplexity after Epoch 15: 124.13
Epoch 16/20
Validation Perplexity after Epoch 16: 120.77
Epoch 17/20
Validation Perplexity after Epoch 17: 118.81
Epoch 18/20
Validation Perplexity after Epoch 18:

  0%|          | 0/5000 [00:00<?, ?it/s]

saved lstm_predictions.npy


### Submission

Upload a submission with the following files to Gradescope:
* proj_1.ipynb (rename to match this exactly)
* lstm_predictions.npy (this should also include all improvements from your exploration)
* neural_trigram_predictions.npy
* bigram_predictions.npy
* report.pdf

You can upload files individually or as part of a zip file, but if using a zip file be sure you are zipping the files directly and not a folder that contains them.

Be sure to check the output of the autograder after it runs.  It should confirm that no files are missing and that the output files have the correct format.  Note that the test set perplexities shown by the autograder are on a completely different scale from your validation set perplexities due to truncating the distribution and selecting different text.  Don't worry if the values seem much worse.